# Find all mathing PMIDs
- had DRUG, but no DISEASE NER

In [23]:
import os
import pandas as pd
import ast
from tqdm import tqdm
import re
from collections import defaultdict
import numpy as np

In [5]:
csv_directory = "/shares/animalwelfare.crs.uzh/preclin_ner_full_ds/model_predictions"  # Adjust as needed
drug_only_rows = []

# Helper function to check for DRUG but not DISEASE
def has_drug_not_disease(ner_string):
    try:
        if pd.isna(ner_string) or not isinstance(ner_string, str):
            return False
        entities = ast.literal_eval(ner_string.strip())
        has_drug = any(ent[2] == 'DRUG' for ent in entities if isinstance(ent, tuple) and len(ent) > 2)
        has_disease = any(ent[2] == 'DISEASE' for ent in entities if isinstance(ent, tuple) and len(ent) > 2)
        return has_drug and not has_disease
    except Exception as e:
        print(f"⚠️ Error parsing: {ner_string}\n{e}")
        return False

# Helper function to check for DRUG but not DISEASE
def has_drug_and_disease(ner_string):
    try:
        if pd.isna(ner_string) or not isinstance(ner_string, str):
            return False
        entities = ast.literal_eval(ner_string.strip())
        has_drug = any(ent[2] == 'DRUG' for ent in entities if isinstance(ent, tuple) and len(ent) > 2)
        has_disease = any(ent[2] == 'DISEASE' for ent in entities if isinstance(ent, tuple) and len(ent) > 2)
        return has_drug and has_disease
    except Exception as e:
        print(f"⚠️ Error parsing: {ner_string}\n{e}")
        return False

# Loop through all CSVs
csv_files = [f for f in os.listdir(csv_directory) if f.endswith(".csv")]
print(f"Processing {len(csv_files)} CSV files...\n")

for filename in tqdm(csv_files, desc="Scanning for DRUG-only rows"):
    filepath = os.path.join(csv_directory, filename)

    try:
        df = pd.read_csv(filepath)

        if 'ner_prediction_BioLinkBERT-base_normalized' not in df.columns:
            print("columnd with predictions not found")
            continue

        df['source_file'] = filename
        filtered = df[df['ner_prediction_BioLinkBERT-base_normalized'].apply(has_drug_and_disease)]

        if not filtered.empty:
            drug_only_rows.append(filtered)

    except Exception as e:
        tqdm.write(f"Error reading {filename}: {e}")

# Combine all DRUG-only rows
if drug_only_rows:
    result_df = pd.concat(drug_only_rows, ignore_index=True)
    print(f"\n✅ Done. Found {len(result_df)} DRUG-only rows.")
else:
    result_df = pd.DataFrame()
    print("\n⚠️ Done. No DRUG-only rows found.")

Processing 618 CSV files...



Scanning for DRUG-only rows: 100%|██████████| 618/618 [04:18<00:00,  2.39it/s] 


✅ Done. Found 596702 DRUG-only rows.


In [6]:
result_df.head()

,PMID,ner_prediction_BioLinkBERT-base_normalized,source_file
0,22274399,"[(88, 100, 'DISEASE', 'glioblastoma'), (147, 1...",test_annotated_BioLinkBERT-base_tuples_2025032...
1,22274590,"[(143, 157, 'DISEASE', 'breast cancers'), (807...",test_annotated_BioLinkBERT-base_tuples_2025032...
2,22274705,"[(26, 34, 'DRUG', 'primycin'), (159, 178, 'DRU...",test_annotated_BioLinkBERT-base_tuples_2025032...
3,22274731,"[(34, 55, 'DISEASE', 'myocardial infarction'),...",test_annotated_BioLinkBERT-base_tuples_2025032...
4,22274733,"[(582, 591, 'DRUG', 'metformin'), (1245, 1254,...",test_annotated_BioLinkBERT-base_tuples_2025032...


In [7]:
result_df.to_csv("/shares/animalwelfare.crs.uzh/preclin_ner_full_ds/model_predictions/drug_and_disease.csv",index=False)

In [8]:
result_df = pd.read_csv("/shares/animalwelfare.crs.uzh/preclin_ner_full_ds/model_predictions/drug_and_disease.csv")

In [9]:
target_pmids = result_df['PMID'].astype(str).unique()
len(target_pmids)

582296

# Get content (title/abstract)

In [10]:
new_folder = "/shares/animalwelfare.crs.uzh/preclin_ner_full_ds/pubmed_animal_docs_large"
filtered_dfs = []

csv_files = [f for f in os.listdir(new_folder) if f.endswith(".csv")]
print(f"Scanning {len(csv_files)} files for matching PMIDs...\n")

for filename in tqdm(csv_files, desc="Filtering new data"):
    filepath = os.path.join(new_folder, filename)

    try:
        df = pd.read_csv(filepath)

        if 'PMID' not in df.columns:
            continue

        df['PMID'] = df['PMID'].astype(str)
        matches = df[df['PMID'].isin(target_pmids)].copy()

        if not matches.empty:
            matches['source_file'] = filename
            filtered_dfs.append(matches)

    except Exception as e:
        tqdm.write(f"Error in {filename}: {e}")

if filtered_dfs:
    combined_text_df = pd.concat(filtered_dfs, ignore_index=True)
    print(f"\n✅ Found {len(combined_text_df)} matching rows across new files.")
else:
    combined_text_df = pd.DataFrame()
    print("\n⚠️ No matching PMIDs found in the new files.")

Scanning 618 files for matching PMIDs...



Filtering new data:  53%|█████▎    | 325/618 [01:44<01:40,  2.90it/s]/sctmp/sdonev/ipykernel_866912/728295467.py:11: DtypeWarning: Columns (1,3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filepath)
Filtering new data: 100%|██████████| 618/618 [03:45<00:00,  2.74it/s]


✅ Found 596702 matching rows across new files.


In [11]:
combined_text_df.head()

,PMID,Text,source_file
0,19118093,Trauma-hemorrhagic shock-induced pulmonary epi...,pubmed_filtered_animal_for_NER_chunk_523.csv
1,19118098,Role of central nervous system aldosterone syn...,pubmed_filtered_animal_for_NER_chunk_523.csv
2,19118113,Early but not late administration of glucagon-...,pubmed_filtered_animal_for_NER_chunk_523.csv
3,19118134,Synergy between enzyme inhibitors of fatty aci...,pubmed_filtered_animal_for_NER_chunk_523.csv
4,19118162,Nox4 NADPH oxidase mediates oxidative stress a...,pubmed_filtered_animal_for_NER_chunk_523.csv


In [12]:
combined_text_df.to_csv("/shares/animalwelfare.crs.uzh/preclin_ner_full_ds/pubmed_animal_docs_large/drug_and_disease_content.csv",index=False)

In [13]:
combined_text_df = pd.read_csv("/shares/animalwelfare.crs.uzh/preclin_ner_full_ds/pubmed_animal_docs_large/drug_and_disease_content.csv")

In [14]:
combined_text_df.head()

,PMID,Text,source_file
0,19118093,Trauma-hemorrhagic shock-induced pulmonary epi...,pubmed_filtered_animal_for_NER_chunk_523.csv
1,19118098,Role of central nervous system aldosterone syn...,pubmed_filtered_animal_for_NER_chunk_523.csv
2,19118113,Early but not late administration of glucagon-...,pubmed_filtered_animal_for_NER_chunk_523.csv
3,19118134,Synergy between enzyme inhibitors of fatty aci...,pubmed_filtered_animal_for_NER_chunk_523.csv
4,19118162,Nox4 NADPH oxidase mediates oxidative stress a...,pubmed_filtered_animal_for_NER_chunk_523.csv


# Run Model KW to Disease Map

In [15]:
text_df = combined_text_df.copy()

In [16]:
text_df.shape

(596702, 3)

In [127]:
# Phrase mapping dictionary 
regex_disease_models = [

    # --- Multiple Sclerosis and Experimental Demyelination Models ---
    (
        re.compile(r"""
            \b(?:
                experimental\s+(?:autoimmune|allergic)\s+encephalomyelitis|
                spontaneous\s+chronic\s+encephalomyelitis|
                opticospinal\s+EAE(?:\s*\(OSE\))?|  # opticospinal EAE (OSE)
                EAE\b|                              # generic EAE
                Experimental\s+Demyelinating\s+Disease|
                experimentally\s+demyelinated|        # explicit coverage
                Theiler(?:'s)?\s+(?:murine\s+)?encephalomyelitis\s+virus(?:-induced)?(?:\s+demyelinating\s+disease)?|
                TMEV-IDD|                            # shorthand
                Theiler's\s+virus-induced\s+demyelinating\s+disease|
                cuprizone(?:-induced)?(?:\s+(?:animal\s+model|CNS\s+))?demyelination(?:\s+model)?|
                cuprizone\s+model(?:\s+of\s+demyelination)?|
                cuprizone\s+and\s+lysolecithin\s*\(?LPC\)?\s+demyelination\s+models|
                lysolecithin-induced\s+(?:demyelination|de-/remyelination)\s+model|
                lysolecithin-induced\s+demyelination|
                demyelinated\s+with\s+ethidium\s+bromide|
                toxic\s+demyelination|
                experimental\s+spinal\s+cord\s+demyelination|
                model\s+of\s+demyelinating\s+disease
            )\b
        """, re.I | re.X),
        "multiple sclerosis"
    ),
    

    # --- PARKINSON’S DISEASE – NEUROTOXIC MODELS (only when used in model context) ---
    #|lipopolysaccharide|lps |rotenone -> seems to give many false positives
    (
        re.compile(r"""
            \b
            (?:
                # All compounds require a suffix:
                mptp
                |1-methyl-4-phenyl-1,2,3,6-tetrahydropyridine
                |6[-\s]?hydroxy[-\s]?dopamine
                |6[-\s]?ohda
                
            )
            [-–\s]+
            (?:induced|exposed|lesioned|injected|treated|model)
            \b
        """, re.I | re.X),
        "Parkinson disease"
    ),
    (
        re.compile(r"\b(?:hemi[-–]?parkinsonian|hemiparkinsonian)\b", re.I),
        "Parkinson disease"
    ),
    (
        re.compile(r"\bTremulous\s+jaw\s+movements?|TBZ-induced|Tremulous\s+jaw\s+movement\s+model\b", re.I),
        "Parkinson disease"
    ),
    (
        re.compile(r"\bMitoPark\b", re.I),
        "Parkinson disease"
    ),
    (
        re.compile(r"\b(?:a-?synuclein\s+mouse\s+model|synuclein\s+mouse\s+model|a-?synuclein\s+rat\s+model|synuclein\s+marmoset\s+model|synuclein\s+monkey\s+model)\b", re.I),
        "Parkinson disease"
    ),


   # --- Alzheimer’s disease models ---
    (
        re.compile(r"""
            \b(
                pdapp |
                tg2576 |
                app23 |
                j20 |
                (app[\-/]?)ps1[0-9a-z]* |
                appswe[\-/]?ps1d?e9 |
                tgswdi |
                tgf344[-\s]?ad |
                3x[tT]g(-ad)? |
                5xfad |
                app[e‚e]693[δdΔ]?-?tg |
                app[-\s]?knock[-\s]?in[-\s]?nl[-\s]?g[-\s]?f |
                mcgill[-\s]?r[-\s]?thy1[-\s]?app
            )\b
        """, re.X),
        "Alzheimer disease"
    )
]


# ──────────────────────────────────────────────────────────────────────────────
# 2.  EXTRACTION FUNCTION – returns a list of tuples exactly like:
#       (start, end, 'DISEASE', raw_phrase)
#       (start, end, 'DISEASE', normalised_name)
# ──────────────────────────────────────────────────────────────────────────────
def extract_neuro_disease_models(text: str):
    """Return list of NER-style tuples for well-established neuro-disease models."""
    text = str(text)
    results = []

    for rgx, disease_name in regex_disease_models:
        for m in rgx.finditer(text):
            start, end = m.start(), m.end()
            raw_phrase = m.group(0)
            # raw match
            results.append((start, end, 'DISEASE_model', raw_phrase))
            # normalised mapping
            results.append((start, end, 'DISEASE', disease_name))

    return results

In [128]:
# ──────────────────────────────────────────────────────────────────────────────
# 3.  APPLY TO YOUR DATAFRAME
# ──────────────────────────────────────────────────────────────────────────────
tqdm.pandas(desc="Extracting disease models")
text_df['disease_mentions_from_model'] = text_df['Text'].progress_apply(extract_neuro_disease_models)


Extracting disease models: 100%|██████████| 596702/596702 [03:41<00:00, 2692.27it/s]


In [129]:
text_df.head()

,PMID,Text,source_file,disease_mentions_from_model
0,19118093,Trauma-hemorrhagic shock-induced pulmonary epi...,pubmed_filtered_animal_for_NER_chunk_523.csv,[]
1,19118098,Role of central nervous system aldosterone syn...,pubmed_filtered_animal_for_NER_chunk_523.csv,[]
2,19118113,Early but not late administration of glucagon-...,pubmed_filtered_animal_for_NER_chunk_523.csv,[]
3,19118134,Synergy between enzyme inhibitors of fatty aci...,pubmed_filtered_animal_for_NER_chunk_523.csv,[]
4,19118162,Nox4 NADPH oxidase mediates oxidative stress a...,pubmed_filtered_animal_for_NER_chunk_523.csv,[]


In [130]:
def parse_cell(cell):
    if isinstance(cell, str):
        try:
            return ast.literal_eval(cell)
        except (ValueError, SyntaxError):
            return []
    elif isinstance(cell, (list, np.ndarray)):
        return list(cell)
    else:
        return []

# 3) Apply it
text_df["disease_mentions_from_model"] = text_df["disease_mentions_from_model"].apply(parse_cell)

# 4) Filter to only rows where the list isn’t empty
df_disease_found = text_df[text_df["disease_mentions_from_model"].map(len) > 0]

In [131]:
df_disease_found.shape

(8461, 4)

In [132]:
pmid_file = "./HERMES_INCLUDED_PMIDs_valid_title.csv" #"./missing_ner_filter_CONTENT.csv"  # <-- Change this
pmid_df = pd.read_csv(pmid_file) #pmid_df = pd.read_csv(pmid_file, sep='|')  # Use sep based on file format
target_pmids = set(pmid_df['pmid'].astype(str))

In [133]:
len(target_pmids)

463

In [134]:
df_disease_found_pmids = set(df_disease_found['PMID'].astype(str))

In [135]:
df_disease_found.head()

,PMID,Text,source_file,disease_mentions_from_model
20,29079063,Neuroprotective and Neuro-restorative Effects ...,pubmed_filtered_animal_for_NER_chunk_523.csv,"[(91, 114, DISEASE_model, 6-Hydroxydopamine Mo..."
387,19010365,Neurochemical changes in the striatum of dyski...,pubmed_filtered_animal_for_NER_chunk_523.csv,"[(981, 1006, DISEASE_model, 6-hydroxydopamine-..."
467,19059812,Inhibitory effect of dimethylthiourea on rat u...,pubmed_filtered_animal_for_NER_chunk_523.csv,"[(1298, 1312, DISEASE_model, 6-OHDA induced), ..."
587,8419792,The possible role of iron in the etiopathology...,pubmed_filtered_animal_for_NER_chunk_523.csv,"[(1619, 1633, DISEASE_model, 6-OHDA-induced), ..."
661,17159262,Neuroprotection of the nigrostriatal dopaminer...,pubmed_filtered_animal_for_NER_chunk_523.csv,"[(1491, 1506, DISEASE_model, 6-OHDA lesioned),..."


In [136]:
len(target_pmids & df_disease_found_pmids)

360

In [137]:
mentions_df = (
    text_df[['PMID', 'disease_mentions_from_model']]
    .explode('disease_mentions_from_model')
    .dropna(subset=['disease_mentions_from_model'])
)
mentions_df['label'] = mentions_df['disease_mentions_from_model'].apply(lambda t: t[2])
mentions_df['value'] = mentions_df['disease_mentions_from_model'].apply(lambda t: t[3])

# Create mapping of PMID to normalized diseases
norm_diseases = (
    mentions_df[mentions_df['label'] == 'DISEASE']
    .groupby('PMID')['value']
    .unique()
    .to_dict()
)

# Filter only raw model phrases
model_mentions_df = mentions_df[mentions_df['label'] == 'DISEASE_model'].copy()

# Add normalized disease from PMID mapping
model_mentions_df['diseases_from_same_doc'] = model_mentions_df['PMID'].map(norm_diseases)

model_mentions_df = model_mentions_df.explode('diseases_from_same_doc')

model_summary = (
    model_mentions_df
    .groupby(['diseases_from_same_doc', 'value'])['PMID']
    .nunique()
    .reset_index(name='unique_PMID_count')
    .sort_values(by=['diseases_from_same_doc', 'unique_PMID_count'], ascending=[True, False])
)
pmid_samples = (
    model_mentions_df
    .groupby(['diseases_from_same_doc', 'value'])['PMID']
    .apply(lambda pmids: list(pd.Series(pmids.unique()).sample(
        n=min(3, len(pmids.unique())),
        random_state=42,
        replace=False
    )) if len(pmids.unique()) > 0 else [])
    .reset_index(name='PMID_sample')
)

# Merge with summary table
model_summary = model_summary.merge(pmid_samples, on=['diseases_from_same_doc', 'value'])
model_summary

,diseases_from_same_doc,value,unique_PMID_count,PMID_sample
0,Alzheimer disease,3xTg,396,"[21782947, 37022634, 28396259]"
1,Alzheimer disease,tg2576,8,"[17021402, 23499467, 24746365]"
2,Alzheimer disease,3xtg,1,[32765947]
3,Parkinson disease,MPTP-induced,1037,"[27115420, 36818224, 23933227]"
4,Parkinson disease,6-OHDA-induced,535,"[26542606, 12435803, 35603858]"
...,...,...,...,...
147,multiple sclerosis,Toxic demyelination,1,[34589847]
148,multiple sclerosis,cuprizone and lysolecithin (LPC) demyelination...,1,[21740973]
149,multiple sclerosis,demyelinated with ethidium bromide,1,[17574629]
150,multiple sclerosis,experimental demyelinating disease,1,[21326918]


In [138]:
model_summary.to_csv("./mapped_disease_from_animal_model_v_drug_and_disease_NER.csv")

In [139]:
df_disease_found.shape

(8461, 4)

In [140]:
df_disease_found.head()

,PMID,Text,source_file,disease_mentions_from_model
20,29079063,Neuroprotective and Neuro-restorative Effects ...,pubmed_filtered_animal_for_NER_chunk_523.csv,"[(91, 114, DISEASE_model, 6-Hydroxydopamine Mo..."
387,19010365,Neurochemical changes in the striatum of dyski...,pubmed_filtered_animal_for_NER_chunk_523.csv,"[(981, 1006, DISEASE_model, 6-hydroxydopamine-..."
467,19059812,Inhibitory effect of dimethylthiourea on rat u...,pubmed_filtered_animal_for_NER_chunk_523.csv,"[(1298, 1312, DISEASE_model, 6-OHDA induced), ..."
587,8419792,The possible role of iron in the etiopathology...,pubmed_filtered_animal_for_NER_chunk_523.csv,"[(1619, 1633, DISEASE_model, 6-OHDA-induced), ..."
661,17159262,Neuroprotection of the nigrostriatal dopaminer...,pubmed_filtered_animal_for_NER_chunk_523.csv,"[(1491, 1506, DISEASE_model, 6-OHDA lesioned),..."


## Merge into existing NER extractions -> already normalized annotations present -> just add the newly found mapping

In [141]:
mondo_mappings = {"Alzheimer disease": "MONDO:0004975",
                  "Parkinson disease": "MONDO:0005180",
                  "multiple sclerosis": "MONDO:0005301"}

In [142]:
already_mapped_terms = pd.read_csv("./ner_mapped_normalized/mapped_preclinical_data.csv")
already_mapped_terms.shape

(548975, 10)

#### exclude rows which have been already correctly mapped

In [143]:
df_disease_found_pmids = set(df_disease_found['PMID'].astype(str))
len(df_disease_found_pmids)

8264

In [144]:
already_mapped_terms['PMID'] = already_mapped_terms['PMID'].astype(str)

# Filter df_other to keep only rows whose PMID is in df_disease_found_pmids
already_mapped_terms_filtered = already_mapped_terms[already_mapped_terms['PMID'].isin(df_disease_found_pmids)]
already_mapped_terms_filtered = already_mapped_terms_filtered[['PMID','disease_term_mondo_norm','disease_mondo_termid']]
already_mapped_terms_filtered.shape

(7842, 3)

In [145]:
# split the pipe-separated fields into lists
def has_correct_mapping(norm, ids):
    terms = norm.split("|")
    mondo_ids = ids.split("|")
    for term, mid in zip(terms, mondo_ids):
        if term in mondo_mappings and mondo_mappings[term] == mid:
            return True
    return False

# apply and filter out
mask = already_mapped_terms_filtered.apply(
    lambda row: not has_correct_mapping(row["disease_term_mondo_norm"],
                                        row["disease_mondo_termid"]),
    axis=1
)
df_to_curate = already_mapped_terms_filtered[mask].reset_index(drop=True)
df_to_curate.shape

(1011, 3)

In [146]:
df_to_curate_pmids = set(df_to_curate['PMID'].astype(str))


In [161]:
df_disease_found[df_disease_found['PMID']==2162953]

,PMID,Text,source_file,disease_mentions_from_model
407950,2162953,Role of opiates in striatal D-1 dopamine recep...,pubmed_filtered_animal_for_NER_chunk_604.csv,"[(804, 830, DISEASE_model, 6-hydroxydopamine-l..."


#### map to mondo format

In [148]:
def extract_mondo_fields(mentions):
    # keep only tuples whose 3rd field contains 'DISEASE'
    disease_mentions = [m for m in mentions if 'DISEASE' in m[2]]
    
    # for each mention text, look up mapping if available
    terms, ids = [], []
    for _, _, _, mention in disease_mentions:
        # strip trailing punctuation
        norm = mention.rstrip(' .;:')
        if norm in mondo_mappings:
            terms.append(norm)
            ids.append(mondo_mappings[norm])
    if terms:
        terms = list(set(terms))
        ids = list(set(ids))
        return "|".join(terms), "|".join(ids)
    else:
        return "", ""

# apply row-wise
result_new_diseases = df_disease_found.copy()
#result_new_diseases = result_new_diseases[result_new_diseases['PMID'].isin(df_to_curate_pmids)]

result_new_diseases[['disease_term_mondo_norm_from_model', 'disease_mondo_termid_from_model']] = result_new_diseases['disease_mentions_from_model'] \
    .apply(lambda mentions: pd.Series(extract_mondo_fields(mentions)))
result_new_diseases = result_new_diseases[['PMID','disease_term_mondo_norm_from_model', 'disease_mondo_termid_from_model']]
result_new_diseases = result_new_diseases.drop_duplicates()
result_new_diseases.shape

/sctmp/sdonev/ipykernel_866912/800950409.py:25: FutureWarning: Returning a DataFrame from Series.apply when the supplied function returns a Series is deprecated and will be removed in a future version.
  .apply(lambda mentions: pd.Series(extract_mondo_fields(mentions)))


(8264, 3)

In [149]:
result_new_diseases['PMID'] = result_new_diseases['PMID'].astype(str)


In [150]:
result_new_diseases.head()

,PMID,disease_term_mondo_norm_from_model,disease_mondo_termid_from_model
20,29079063,Parkinson disease,MONDO:0005180
387,19010365,Parkinson disease,MONDO:0005180
467,19059812,Parkinson disease,MONDO:0005180
587,8419792,Parkinson disease,MONDO:0005180
661,17159262,Parkinson disease,MONDO:0005180


In [151]:
df_to_curate_merged = df_to_curate.merge(result_new_diseases, on="PMID", how="left")

In [152]:
df_to_curate_merged.shape

(1011, 5)

In [156]:
df_to_curate_merged_pmids = set(df_to_curate_merged['PMID'].unique())
len(df_to_curate_merged_pmids)

1011

In [157]:
len(target_pmids & df_to_curate_merged_pmids)

52

In [153]:
df_to_curate_merged.head(10)

,PMID,disease_term_mondo_norm,disease_mondo_termid,disease_term_mondo_norm_from_model,disease_mondo_termid_from_model
0,31739156,relapsing-remitting multiple sclerosis,MONDO:0005314,multiple sclerosis,MONDO:0005301
1,2162953,dopamine receptor supersensitivity,-1,Parkinson disease,MONDO:0005180
2,21037084,neurodegenerative disease,MONDO:0005559,multiple sclerosis,MONDO:0005301
3,2790469,parkinsonian disorder,MONDO:0021095,Parkinson disease,MONDO:0005180
4,14559299,parkinsonian disorder|neurodegenerative disease,MONDO:0021095|MONDO:0005559,Parkinson disease,MONDO:0005180
5,24670802,lupus erythematosus|disease,MONDO:0004670|MONDO:0000001,multiple sclerosis,MONDO:0005301
6,23286644,oxidative stress,-1,Parkinson disease,MONDO:0005180
7,17891523,myocardial ischemia|reperfusion-induced necrosis,MONDO:0024644|-1,Parkinson disease,MONDO:0005180
8,10403433,movement disorder,MONDO:0005395,Parkinson disease,MONDO:0005180
9,6149306,self-mutilation behavior,-1,Parkinson disease,MONDO:0005180


In [160]:
df_to_curate_merged['disease_term_mondo_norm']= df_to_curate_merged['disease_term_mondo_norm']+"|"+df_to_curate_merged['disease_term_mondo_norm_from_model']
df_to_curate_merged['disease_mondo_termid']= df_to_curate_merged['disease_mondo_termid']+"|"+df_to_curate_merged['disease_mondo_termid_from_model']
df_to_curate_merged.head()

,PMID,disease_term_mondo_norm,disease_mondo_termid,disease_term_mondo_norm_from_model,disease_mondo_termid_from_model
0,31739156,relapsing-remitting multiple sclerosis|multipl...,MONDO:0005314|MONDO:0005301,multiple sclerosis,MONDO:0005301
1,2162953,dopamine receptor supersensitivity|Parkinson d...,-1|MONDO:0005180,Parkinson disease,MONDO:0005180
2,21037084,neurodegenerative disease|multiple sclerosis,MONDO:0005559|MONDO:0005301,multiple sclerosis,MONDO:0005301
3,2790469,parkinsonian disorder|Parkinson disease,MONDO:0021095|MONDO:0005180,Parkinson disease,MONDO:0005180
4,14559299,parkinsonian disorder|neurodegenerative diseas...,MONDO:0021095|MONDO:0005559|MONDO:0005180,Parkinson disease,MONDO:0005180


In [162]:
df_to_curate_merged = df_to_curate_merged.drop(
    columns=['disease_term_mondo_norm_from_model',
             'disease_mondo_termid_from_model']
)
df_to_curate_merged.head()

,PMID,disease_term_mondo_norm,disease_mondo_termid
0,31739156,relapsing-remitting multiple sclerosis|multipl...,MONDO:0005314|MONDO:0005301
1,2162953,dopamine receptor supersensitivity|Parkinson d...,-1|MONDO:0005180
2,21037084,neurodegenerative disease|multiple sclerosis,MONDO:0005559|MONDO:0005301
3,2790469,parkinsonian disorder|Parkinson disease,MONDO:0021095|MONDO:0005180
4,14559299,parkinsonian disorder|neurodegenerative diseas...,MONDO:0021095|MONDO:0005559|MONDO:0005180


#### replace in original annotations

In [164]:
already_mapped_terms.shape

(548975, 10)

In [165]:
# rename the new cols so they don’t collide
df_new_ren = df_to_curate_merged.rename(columns={
    'disease_term_mondo_norm':    'new_disease_term',
    'disease_mondo_termid':       'new_disease_id'
})

# left-merge onto your existing df
merged = already_mapped_terms.merge(
    df_new_ren[['PMID','new_disease_term','new_disease_id']],
    on='PMID',
    how='left'
)

# wherever there’s a new value, use it; otherwise keep the old
merged['disease_term_mondo_norm'] = merged['new_disease_term'].fillna(merged['disease_term_mondo_norm'])
merged['disease_mondo_termid']  = merged['new_disease_id'].fillna(merged['disease_mondo_termid'])

(548975, 12)

In [167]:
merged = merged.drop(columns=['new_disease_term','new_disease_id', 'Unnamed: 0'])

# drop the helper cols
merged.shape

(548975, 10)

In [170]:
merged.head()

,PMID,unique_conditions_linkbert_predictions,unique_interventions_linkbert_predictions,linkbert_mapped_conditions,linkbert_mapped_drugs,disease_term_mondo_norm,disease_mondo_termid,drug_term_umls_norm,drug_umls_termid
0,31733831,asthma,isorhynchophylline,asthma,isorhynchophylline,asthma,MONDO:0004979,isorhynchophylline,C0245133
1,31733833,myocardial infarction,antgomir-1192|mir-1192|agomir-1192,myocardial infarction,antgomir-1192|mir-1192|agomir-1192,myocardial infarction,MONDO:0005068,antgomir-1192|mir-1192|agomir-1192,-1|-1|-1
2,31733925,systemic lupus erythematosus,g2|hla-g2,systemic lupus erythematosus,g2|hla-g2,systemic lupus erythematosus,MONDO:0007915,g2|HLA-G2 Isoform,-1|C0967254
3,31733940,cognitive impairment,minocycline,cognitive impairment,minocycline,cognitive disorder,MONDO:0002039,Minocycline,C0026187
4,31734027,oxaliplatin-induced peripheral neuropathy|cumu...,tadalafil|phosphodiesterase type 5 inhibitor t...,oxaliplatin-induced peripheral neuropathy|cumu...,tadalafil|phosphodiesterase type 5 inhibitor t...,oxaliplatin-induced peripheral neuropathy|cumu...,-1|-1|MONDO:0003620,Tadalafil|Tadalafil,C1176316|C1176316


In [172]:
merged[merged['PMID']=="10403433"]

,PMID,unique_conditions_linkbert_predictions,unique_interventions_linkbert_predictions,linkbert_mapped_conditions,linkbert_mapped_drugs,disease_term_mondo_norm,disease_mondo_termid,drug_term_umls_norm,drug_umls_termid
5315,10403433,dyskinesia,d1 and d2 dopaminomimetic|skf 82958|d1 dopamin...,dyskinesias,d1 and d2 dopaminomimetic|skf 82958|d1 dopamin...,movement disorder|Parkinson disease,MONDO:0005395|MONDO:0005180,d1 and d2 dopaminomimetic|SK&F 82958|SK&F 8295...,-1|C0142426|C0142426|-1|C0107994|-1


In [173]:
merged.to_csv('./ner_mapped_normalized/mapped_preclinical_data_enriched.csv', index=False)


## Merge into existing NER extractions -> for further processing ie normalization via dict and embeddings

In [41]:
result_df.head()

,PMID,ner_prediction_BioLinkBERT-base_normalized,source_file
0,22274300,"[(21, 32, 'DRUG', 'fenofibrate'), (107, 126, '...",test_annotated_BioLinkBERT-base_tuples_2025032...
1,22274401,"[(15, 24, 'DRUG', 'vitamin a'), (299, 308, 'DR...",test_annotated_BioLinkBERT-base_tuples_2025032...
2,22274564,"[(1142, 1163, 'DRUG', 'pparα agonist wy14643')...",test_annotated_BioLinkBERT-base_tuples_2025032...
3,22274565,"[(0, 10, 'DRUG', 'estrogenic'), (369, 377, 'DR...",test_annotated_BioLinkBERT-base_tuples_2025032...
4,22274614,"[(86, 91, 'DRUG', 'miraculin -')]",test_annotated_BioLinkBERT-base_tuples_2025032...


In [108]:
df_disease_found_with_drug = df_disease_found[['PMID','disease_mentions_from_model']].merge(result_df[['PMID','ner_prediction_BioLinkBERT-base_normalized']], on="PMID", how='left')
df_disease_found_with_drug = df_disease_found_with_drug.drop_duplicates(subset="PMID")

In [109]:
df_disease_found_with_drug.shape

(4491, 3)

In [110]:
df_disease_found_with_drug.to_csv("./out_model_to_disease_map/df_disease_found_with_drug.csv",index=False)

In [112]:
import ast
import pandas as pd

def safe_convert_to_list(value):
    """Safely convert value to list"""
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        try:
            return ast.literal_eval(value)
        except:
            return []
    return []

# Dictionary to track each disease value and PMIDs
disease_details = defaultdict(lambda: {'count': 0, 'pmids': set()})

def merge_disease_mentions_detailed_tracking(row):
    global disease_details
    
    # Convert both columns to lists safely
    existing_predictions = safe_convert_to_list(row['ner_prediction_BioLinkBERT-base_normalized'])
    disease_mentions = safe_convert_to_list(row['disease_mentions_from_model'])
    pmid = row['PMID']
    
    # Add disease mentions where third element equals 'DISEASE'
    for mention in disease_mentions:
        if len(mention) >= 3 and 'DISEASE' == str(mention[2]):
            existing_predictions.append(mention)
            # Track the disease value (4th element of tuple)
            if len(mention) >= 4:
                disease_value = mention[3]  # Assuming disease name is 4th element
                disease_details[disease_value]['count'] += 1
                disease_details[disease_value]['pmids'].add(pmid)
    
    return existing_predictions

# Apply the function
df_disease_found_with_drug['ner_prediction_BioLinkBERT-base_normalized'] = df_disease_found_with_drug.apply(merge_disease_mentions_detailed_tracking, axis=1)

# Create a summary DataFrame for easier analysis
summary_data = []
for disease, info in disease_details.items():
    summary_data.append({
        'disease': disease,
        'total_mentions': info['count'],
        'unique_pmids': len(info['pmids']),
        'pmids_list': sorted(list(info['pmids']))
    })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values('unique_pmids', ascending=False)

# Print results
print("DISEASE VALUES AND PMID COUNTS:")
print("=" * 80)
for _, row in summary_df.iterrows():
    print(f"Disease: {row['disease']}")
    print(f"  Total mentions: {row['total_mentions']}")
    print(f"  Unique PMIDs: {row['unique_pmids']}")
    print("-" * 60)

print(f"\nSUMMARY STATISTICS:")
print(f"Total unique diseases: {len(summary_df)}")
print(f"Total disease mentions added: {summary_df['total_mentions'].sum()}")
print(f"Total unique PMIDs with diseases: {len(set().union(*summary_df['pmids_list']))}")

# Show diseases sorted by number of PMIDs
print(f"\nTOP DISEASES BY NUMBER OF PMIDs:")
print("=" * 50)
print(summary_df[['disease', 'unique_pmids', 'total_mentions']].to_string(index=False))


DISEASE VALUES AND PMID COUNTS:
Disease: parkinson's disease
  Total mentions: 2492
  Unique PMIDs: 1631
------------------------------------------------------------
Disease: alzheimer's disease
  Total mentions: 1790
  Unique PMIDs: 1611
------------------------------------------------------------
Disease: multiple sclerosis
  Total mentions: 5748
  Unique PMIDs: 1413
------------------------------------------------------------

SUMMARY STATISTICS:
Total unique diseases: 3
Total disease mentions added: 10030
Total unique PMIDs with diseases: 4491

TOP DISEASES BY NUMBER OF PMIDs:
            disease  unique_pmids  total_mentions
parkinson's disease          1631            2492
alzheimer's disease          1611            1790
 multiple sclerosis          1413            5748


In [113]:
df_disease_found_with_drug.iloc[3]['ner_prediction_BioLinkBERT-base_normalized']

[(46, 54, 'DRUG', 'chalcone'),
 (259, 267, 'DRUG', 'chalcone'),
 (814, 820, 'DRUG', 'dt ch )'),
 (1508, 1519, 'DISEASE', "alzheimer's disease")]

In [114]:
df_disease_found_with_drug[['PMID','ner_prediction_BioLinkBERT-base_normalized']].to_csv("./out_model_to_disease_map/from_model_disease_with_drug.csv",index=False)

In [115]:
df_disease_found_with_drug.PMID.nunique()

4491